# 🏀 Peak, Prime & Decline: NBA Career Trajectory Modeling

> **When do NBA players peak? Does it vary by skill? We model aging curves across 20+ seasons of data, predict peak performance from early-career stats, and identify who ages most gracefully.**

**Part 9 of 10** in the [wyattowalsh/basketball](https://www.kaggle.com/datasets/wyattowalsh/basketball) companion notebook series.

---

## What makes this analysis unique?

Most aging curve studies rely on simple averages. This notebook goes further with:

- **Parametric curve fitting** (Gaussian aging model via `scipy.optimize.curve_fit`)
- **Position-specific aging** — guards, forwards, and centers age differently
- **Era comparison** — are modern players lasting longer than prior generations?
- **Peak prediction** — RandomForest model predicts ceiling from early-career stats
- **Active player projections** — fitted career curves extrapolated 5 seasons forward
- **Aging gracefully index** — who most defies their expected decline?

---

## Table of Contents

1. [Setup & Data Loading](#1)
2. [Raw Aging Curves](#2)
3. [Position-Specific Curves](#3)
4. [Parametric Aging Model](#4)
5. [Era Comparison](#5)
6. [Peak Season Prediction](#6)
7. [Active Player Projections](#7)
8. [Aging Gracefully Index](#8)
9. [Conclusion & Cross-Links](#9)

<a id="1"></a>
## 1. Setup & Data Loading

In [ ]:
!pip install -q "duckdb>=1.4,<2" "polars>=1.38,<2" "plotly>=5.18,<6" "scikit-learn>=1.4,<2" "scipy>=1.12,<2"

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

from scipy.optimize import curve_fit
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore", category=FutureWarning)

# Import shared utilities
_kaggle_path = Path("/kaggle/input/basketball")
sys.path.insert(0, str(_kaggle_path if _kaggle_path.exists() else Path(".")))
from nbadb_utils import COLORS, TEMPLATE, takeaway, get_connection

conn = get_connection()

In [ ]:
# --- Load core aging dataset ---
aging_sql = """
SELECT
    aps.player_id,
    dp.full_name AS player_name,
    dp.position,
    dp.birth_date,
    dp.draft_year,
    dp.draft_round,
    dp.draft_number,
    dp.is_active,
    aps.season_year,
    aps.season_type,
    aps.gp,
    aps.avg_min,
    aps.avg_pts,
    aps.avg_reb,
    aps.avg_ast,
    aps.avg_stl,
    aps.avg_blk,
    aps.fg_pct,
    aps.fg3_pct,
    aps.ft_pct,
    aps.avg_ts_pct,
    aps.avg_usg_pct,
    aps.avg_pie,
    -- Derive player age as approximate mid-season age (using end year of season)
    (CAST(LEFT(aps.season_year, 4) AS INT) + 1) - EXTRACT(YEAR FROM CAST(dp.birth_date AS DATE)) AS player_age
FROM agg_player_season aps
JOIN dim_player dp ON aps.player_id = dp.player_id
WHERE aps.season_type = 'Regular Season'
  AND aps.gp >= 20
  AND dp.birth_date IS NOT NULL
ORDER BY dp.full_name, aps.season_year
"""

aging_df = conn.sql(aging_sql).pl()
print(f"Aging dataset: {aging_df.shape[0]:,} player-seasons, {aging_df.shape[1]} columns")
print(f"Unique players: {aging_df['player_id'].n_unique():,}")
print(f"Age range: {aging_df['player_age'].min()} - {aging_df['player_age'].max()}")
print(f"Seasons: {aging_df['season_year'].min()} to {aging_df['season_year'].max()}")
aging_df.head(5)

<a id="2"></a>
## 2. Raw Aging Curves

How do key stats evolve with age across the entire NBA population? We compute
the mean and standard deviation of scoring, playmaking, rebounding, steals, and
blocks by player age, then overlay individual career paths for five all-time greats.

In [ ]:
# Compute population aging curves (ages 19-40)
age_range = aging_df.filter(
    (pl.col("player_age") >= 19) & (pl.col("player_age") <= 40)
)

stats_to_plot = [
    ("avg_pts", "Points Per Game"),
    ("avg_ast", "Assists Per Game"),
    ("avg_reb", "Rebounds Per Game"),
    ("avg_stl", "Steals Per Game"),
    ("avg_blk", "Blocks Per Game"),
]

stat_colors = {
    "avg_pts": COLORS["gold"],
    "avg_ast": COLORS["blue"],
    "avg_reb": COLORS["green"],
    "avg_stl": COLORS["red"],
    "avg_blk": COLORS["purple"],
}

age_stats = (
    age_range
    .group_by("player_age")
    .agg([
        pl.col("avg_pts").mean().alias("mean_pts"),
        pl.col("avg_pts").std().alias("std_pts"),
        pl.col("avg_ast").mean().alias("mean_ast"),
        pl.col("avg_ast").std().alias("std_ast"),
        pl.col("avg_reb").mean().alias("mean_reb"),
        pl.col("avg_reb").std().alias("std_reb"),
        pl.col("avg_stl").mean().alias("mean_stl"),
        pl.col("avg_stl").std().alias("std_stl"),
        pl.col("avg_blk").mean().alias("mean_blk"),
        pl.col("avg_blk").std().alias("std_blk"),
        pl.col("player_id").count().alias("n_players"),
    ])
    .sort("player_age")
)

print("Population aging curves computed.")
print(f"Sample sizes by age (min/max): {age_stats['n_players'].min()} / {age_stats['n_players'].max()}")

In [ ]:
# All-time greats to overlay
greats = {
    "LeBron James": {"color": "#552583", "symbol": "circle"},
    "Kobe Bryant": {"color": "#FDB927", "symbol": "diamond"},
    "Tim Duncan": {"color": "#C4CED4", "symbol": "square"},
    "Stephen Curry": {"color": "#1D428A", "symbol": "triangle-up"},
    "Kevin Durant": {"color": "#007A33", "symbol": "cross"},
}

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[title for _, title in stats_to_plot] + [""],
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
)

ages = age_stats["player_age"].to_list()

for idx, (stat_col, stat_title) in enumerate(stats_to_plot):
    row = idx // 2 + 1
    col = idx % 2 + 1
    stat_short = stat_col.replace("avg_", "")
    means = age_stats[f"mean_{stat_short}"].to_list()
    stds = age_stats[f"std_{stat_short}"].to_list()

    upper = [m + s for m, s in zip(means, stds)]
    lower = [m - s for m, s in zip(means, stds)]

    # Error band (upper)
    fig.add_trace(go.Scatter(
        x=ages, y=upper,
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip",
    ), row=row, col=col)

    # Error band (lower, filled to upper)
    fig.add_trace(go.Scatter(
        x=ages, y=lower,
        mode="lines", line=dict(width=0),
        fill="tonexty",
        fillcolor=f"rgba(100,100,100,0.15)",
        showlegend=False, hoverinfo="skip",
    ), row=row, col=col)

    # Mean line
    fig.add_trace(go.Scatter(
        x=ages, y=means,
        mode="lines",
        line=dict(color=stat_colors[stat_col], width=3),
        name=stat_title,
        showlegend=(idx == 0),
        hovertemplate="Age %{x}<br>Mean: %{y:.2f}<extra></extra>",
    ), row=row, col=col)

    # Overlay individual greats
    for player_name, props in greats.items():
        player_data = age_range.filter(
            pl.col("player_name") == player_name
        ).sort("player_age")
        if player_data.shape[0] == 0:
            continue
        fig.add_trace(go.Scatter(
            x=player_data["player_age"].to_list(),
            y=player_data[stat_col].to_list(),
            mode="markers+lines",
            marker=dict(color=props["color"], symbol=props["symbol"], size=6),
            line=dict(color=props["color"], width=1, dash="dot"),
            name=player_name,
            legendgroup=player_name,
            showlegend=(idx == 0),
            hovertemplate=f"{player_name}<br>Age %{{x}}<br>{stat_title}: %{{y:.1f}}<extra></extra>",
        ), row=row, col=col)

fig.update_layout(
    template=TEMPLATE,
    height=1000,
    width=1000,
    title="NBA Aging Curves: Population Mean (+/- 1 SD) with All-Time Greats",
    showlegend=True,
    legend=dict(x=0.55, y=0.15, font=dict(size=10)),
)
fig.show()

takeaway(
    "Scoring peaks around age 27, while playmaking (assists) peaks slightly later "
    "around age 28-29. Steals and blocks decline earlier and more steeply than "
    "scoring, suggesting that athleticism-dependent skills fade first. LeBron's "
    "scoring trajectory is remarkably flat through age 37 -- a historic outlier."
)

<a id="3"></a>
## 3. Position-Specific Aging Curves

Do guards, forwards, and centers age differently? We group players by position
and compare their aging trajectories for scoring, playmaking, and rebounding.

In [ ]:
# Map positions to three groups
position_map = {
    "Guard": ["G", "PG", "SG", "G-F"],
    "Forward": ["F", "SF", "PF", "F-G", "F-C"],
    "Center": ["C", "C-F"],
}

# Create reverse lookup
pos_lookup = {}
for group, positions in position_map.items():
    for pos in positions:
        pos_lookup[pos] = group

pos_df = age_range.with_columns(
    pl.col("position").replace(pos_lookup, default="Forward").alias("pos_group")
)

pos_colors = {
    "Guard": COLORS["blue"],
    "Forward": COLORS["gold"],
    "Center": COLORS["red"],
}

pos_stats_to_plot = [
    ("avg_pts", "Points Per Game"),
    ("avg_ast", "Assists Per Game"),
    ("avg_reb", "Rebounds Per Game"),
]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[title for _, title in pos_stats_to_plot],
    horizontal_spacing=0.07,
)

for col_idx, (stat_col, stat_title) in enumerate(pos_stats_to_plot, 1):
    for pos_group, color in pos_colors.items():
        pos_age_stats = (
            pos_df.filter(pl.col("pos_group") == pos_group)
            .group_by("player_age")
            .agg([
                pl.col(stat_col).mean().alias("mean_val"),
                pl.col(stat_col).std().alias("std_val"),
                pl.col("player_id").count().alias("n"),
            ])
            .filter(pl.col("n") >= 10)  # require minimum sample
            .sort("player_age")
        )

        ages_pos = pos_age_stats["player_age"].to_list()
        means_pos = pos_age_stats["mean_val"].to_list()
        stds_pos = pos_age_stats["std_val"].to_list()

        upper_pos = [m + s for m, s in zip(means_pos, stds_pos)]
        lower_pos = [max(0, m - s) for m, s in zip(means_pos, stds_pos)]

        # Error band
        fig.add_trace(go.Scatter(
            x=ages_pos, y=upper_pos,
            mode="lines", line=dict(width=0),
            showlegend=False, hoverinfo="skip",
        ), row=1, col=col_idx)
        fig.add_trace(go.Scatter(
            x=ages_pos, y=lower_pos,
            mode="lines", line=dict(width=0),
            fill="tonexty",
            fillcolor=color.replace("#", "rgba(") + ",0.12)" if color.startswith("#") else f"rgba(100,100,100,0.12)",
            showlegend=False, hoverinfo="skip",
        ), row=1, col=col_idx)

        # Mean line
        fig.add_trace(go.Scatter(
            x=ages_pos, y=means_pos,
            mode="lines",
            line=dict(color=color, width=3),
            name=pos_group,
            legendgroup=pos_group,
            showlegend=(col_idx == 1),
            hovertemplate=f"{pos_group}<br>Age %{{x}}<br>{stat_title}: %{{y:.2f}}<extra></extra>",
        ), row=1, col=col_idx)

fig.update_layout(
    template=TEMPLATE,
    height=450,
    width=1100,
    title="Position-Specific Aging Curves: How Guards, Forwards & Centers Age Differently",
    showlegend=True,
)
for i in range(1, 4):
    fig.update_xaxes(title_text="Player Age", row=1, col=i)
fig.show()

takeaway(
    "Guards maintain playmaking ability (assists) deeper into their careers, peaking "
    "around age 29-30. Centers show the earliest and steepest scoring decline, "
    "consistent with their reliance on athleticism near the rim. Forwards occupy "
    "the middle ground. Rebounding for centers stays more stable than their scoring, "
    "suggesting positional rebounding (boxing out, positioning) ages better than "
    "athletic finishing."
)

<a id="4"></a>
## 4. Parametric Aging Model

We fit a Gaussian aging model to each stat:

$$y = a \cdot \exp\left(-\frac{1}{2}\left(\frac{x - \text{peak}}{\text{width}}\right)^2\right) + c$$

Where:
- **peak** = age of maximum performance
- **width** = how quickly performance declines (larger = slower decline)
- **a** = amplitude above baseline
- **c** = baseline level

In [ ]:
def gaussian_aging(x, a, peak, width, c):
    """Gaussian aging curve: performance peaks at `peak` and decays with `width`."""
    return a * np.exp(-0.5 * ((x - peak) / width) ** 2) + c


# Fit per stat
fit_stats = [
    ("avg_pts", "mean_pts", "Points"),
    ("avg_ast", "mean_ast", "Assists"),
    ("avg_reb", "mean_reb", "Rebounds"),
    ("avg_stl", "mean_stl", "Steals"),
    ("avg_blk", "mean_blk", "Blocks"),
]

x_data = np.array(age_stats["player_age"].to_list(), dtype=float)
fit_results = {}

for stat_col, mean_col, label in fit_stats:
    y_data = np.array(age_stats[mean_col].to_list(), dtype=float)
    # Remove NaN
    mask = ~np.isnan(y_data)
    x_fit, y_fit = x_data[mask], y_data[mask]

    try:
        popt, pcov = curve_fit(
            gaussian_aging, x_fit, y_fit,
            p0=[y_fit.max() - y_fit.min(), 27, 5, y_fit.min()],
            bounds=([0, 18, 1, 0], [50, 35, 20, 20]),
            maxfev=10000,
        )
        perr = np.sqrt(np.diag(pcov))
        fit_results[label] = {
            "amplitude": round(popt[0], 2),
            "peak_age": round(popt[1], 1),
            "width": round(popt[2], 1),
            "baseline": round(popt[3], 2),
            "peak_age_err": round(perr[1], 2),
            "popt": popt,
        }
        print(f"{label:10s}: peak_age={popt[1]:.1f} +/- {perr[1]:.2f}, "
              f"width={popt[2]:.1f}, amplitude={popt[0]:.2f}, baseline={popt[3]:.2f}")
    except RuntimeError as e:
        print(f"{label:10s}: curve_fit failed -- {e}")

In [ ]:
# Parameter summary table
param_table = pl.DataFrame([
    {
        "Stat": label,
        "Peak Age": f"{r['peak_age']:.1f} +/- {r['peak_age_err']:.2f}",
        "Width (Decline Rate)": r["width"],
        "Amplitude": r["amplitude"],
        "Baseline": r["baseline"],
    }
    for label, r in fit_results.items()
])

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=list(param_table.columns),
        fill_color=COLORS["purple"],
        font=dict(color="white", size=13),
        align="center",
    ),
    cells=dict(
        values=[param_table[col].to_list() for col in param_table.columns],
        fill_color="white",
        align="center",
        font=dict(size=12),
        height=28,
    ),
)])
fig_table.update_layout(
    title="Parametric Aging Model: Fitted Parameters",
    height=250,
    template=TEMPLATE,
)
fig_table.show()

In [ ]:
# Plot fitted curves overlaid on raw data
x_smooth = np.linspace(19, 40, 200)

fig = go.Figure()

fit_colors = {
    "Points": COLORS["gold"],
    "Assists": COLORS["blue"],
    "Rebounds": COLORS["green"],
    "Steals": COLORS["red"],
    "Blocks": COLORS["purple"],
}

for (stat_col, mean_col, label) in fit_stats:
    if label not in fit_results:
        continue
    y_data = np.array(age_stats[mean_col].to_list(), dtype=float)
    color = fit_colors[label]

    # Raw data points
    fig.add_trace(go.Scatter(
        x=x_data.tolist(), y=y_data.tolist(),
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.6),
        name=f"{label} (raw)",
        legendgroup=label,
        hovertemplate=f"{label}<br>Age %{{x}}<br>Mean: %{{y:.2f}}<extra></extra>",
    ))

    # Fitted curve
    popt = fit_results[label]["popt"]
    y_smooth = gaussian_aging(x_smooth, *popt)
    fig.add_trace(go.Scatter(
        x=x_smooth.tolist(), y=y_smooth.tolist(),
        mode="lines",
        line=dict(color=color, width=3),
        name=f"{label} (fit)",
        legendgroup=label,
        hovertemplate=f"{label} fit<br>Age %{{x:.1f}}<br>Predicted: %{{y:.2f}}<extra></extra>",
    ))

    # Mark peak age
    peak_val = gaussian_aging(popt[1], *popt)
    fig.add_trace(go.Scatter(
        x=[popt[1]], y=[peak_val],
        mode="markers",
        marker=dict(color=color, size=14, symbol="star", line=dict(color="black", width=1)),
        name=f"{label} peak ({popt[1]:.1f})",
        legendgroup=label,
        showlegend=True,
        hovertemplate=f"{label} peak<br>Age: {popt[1]:.1f}<br>Value: {peak_val:.2f}<extra></extra>",
    ))

fig.update_layout(
    template=TEMPLATE,
    title="Parametric Aging Model: Gaussian Fits to NBA Stat Trajectories",
    xaxis_title="Player Age",
    yaxis_title="Per-Game Average",
    height=600,
    legend=dict(x=1.02, y=1, font=dict(size=10)),
)
fig.show()

pts_peak = fit_results.get("Points", {}).get("peak_age", "?")
ast_peak = fit_results.get("Assists", {}).get("peak_age", "?")
stl_peak = fit_results.get("Steals", {}).get("peak_age", "?")
blk_peak = fit_results.get("Blocks", {}).get("peak_age", "?")

takeaway(
    f"The Gaussian model confirms scoring peaks at age ~{pts_peak}, assists at ~{ast_peak}, "
    f"while steals peak earliest at ~{stl_peak} and blocks at ~{blk_peak}. "
    "Athleticism-dependent skills (steals, blocks) have narrower width parameters, "
    "meaning sharper declines. Skill-based stats (assists) have wider curves, "
    "indicating more gradual aging."
)

<a id="5"></a>
## 5. Era Comparison

Are modern players peaking later or lasting longer? We compare aging curves
across draft decades: 2000s (draft years 2000-2009), 2010s (2010-2019),
and 2020s (2020+).

In [ ]:
# Assign draft decade
era_df = age_range.filter(
    pl.col("draft_year").is_not_null()
).with_columns(
    pl.when(pl.col("draft_year").is_between(2000, 2009))
      .then(pl.lit("2000s"))
      .when(pl.col("draft_year").is_between(2010, 2019))
      .then(pl.lit("2010s"))
      .when(pl.col("draft_year") >= 2020)
      .then(pl.lit("2020s"))
      .otherwise(pl.lit("Pre-2000"))
      .alias("draft_decade")
)

# Focus on the three modern decades
era_df = era_df.filter(pl.col("draft_decade").is_in(["2000s", "2010s", "2020s"]))

era_colors = {
    "2000s": COLORS["blue"],
    "2010s": COLORS["gold"],
    "2020s": COLORS["green"],
}

era_stats_to_plot = [
    ("avg_pts", "Scoring (PPG)"),
    ("avg_ast", "Assists (APG)"),
    ("avg_reb", "Rebounds (RPG)"),
]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[title for _, title in era_stats_to_plot],
    horizontal_spacing=0.07,
)

for col_idx, (stat_col, stat_title) in enumerate(era_stats_to_plot, 1):
    for decade, color in era_colors.items():
        era_age = (
            era_df.filter(pl.col("draft_decade") == decade)
            .group_by("player_age")
            .agg([
                pl.col(stat_col).mean().alias("mean_val"),
                pl.col("player_id").count().alias("n"),
            ])
            .filter(pl.col("n") >= 10)
            .sort("player_age")
        )

        fig.add_trace(go.Scatter(
            x=era_age["player_age"].to_list(),
            y=era_age["mean_val"].to_list(),
            mode="lines+markers",
            line=dict(color=color, width=3),
            marker=dict(size=5),
            name=f"{decade} draftees",
            legendgroup=decade,
            showlegend=(col_idx == 1),
            hovertemplate=f"{decade}<br>Age %{{x}}<br>{stat_title}: %{{y:.2f}}<extra></extra>",
        ), row=1, col=col_idx)

# Add vertical dashed lines showing max observed age per era (survivorship bias)
for decade in ["2000s", "2010s", "2020s"]:
    max_age = era_df.filter(pl.col("draft_decade") == decade)["player_age"].max()
    fig.add_vline(x=max_age, line_dash="dash", line_color="gray", opacity=0.5,
                  annotation_text=f"{decade} max age: {max_age}")

fig.update_layout(
    template=TEMPLATE,
    height=450,
    width=1100,
    title="Era Comparison: Aging Curves by Draft Decade",
    showlegend=True,
)
for i in range(1, 4):
    fig.update_xaxes(title_text="Player Age", row=1, col=i)
fig.show()

takeaway(
    "2010s draftees show higher early-career scoring than 2000s draftees at the same "
    "ages, consistent with the pace-and-space revolution increasing offensive output. "
    "The 2020s cohort is truncated at ~age 25-26 — decline patterns cannot be compared "
    "with earlier eras. Modern load management and sports science may extend primes, "
    "but we need more data to confirm."
)

<a id="6"></a>
## 6. Peak Season Prediction

Can we predict a player's career-best scoring season from their first three years?
We use a **RandomForestRegressor** with early-career averages, efficiency metrics,
draft position, and position as features.

In [ ]:
# For each player, compute: peak PPG (excluding first 3 seasons), and their first 3 seasons' averages
peak_sql = """
WITH player_seasons AS (
    SELECT
        aps.player_id,
        dp.full_name,
        dp.position,
        dp.draft_round,
        dp.draft_number,
        aps.season_year,
        aps.avg_pts,
        aps.avg_ast,
        aps.avg_reb,
        aps.fg_pct,
        aps.avg_min,
        ROW_NUMBER() OVER (PARTITION BY aps.player_id ORDER BY aps.season_year) AS season_num
    FROM agg_player_season aps
    JOIN dim_player dp ON aps.player_id = dp.player_id
    WHERE aps.season_type = 'Regular Season'
      AND aps.gp >= 20
),

peak AS (
    SELECT
        player_id,
        MAX(CASE WHEN season_num > 3 THEN avg_pts END) AS peak_ppg
    FROM player_seasons
    GROUP BY player_id
    HAVING COUNT(*) >= 4  -- need at least 4 seasons (3 early + 1 more)
),

early_career AS (
    SELECT
        ps.player_id,
        ps.full_name,
        ps.position,
        ps.draft_round,
        ps.draft_number,
        AVG(CASE WHEN ps.season_num <= 3 THEN ps.avg_pts END) AS early_pts,
        AVG(CASE WHEN ps.season_num <= 3 THEN ps.avg_ast END) AS early_ast,
        AVG(CASE WHEN ps.season_num <= 3 THEN ps.avg_reb END) AS early_reb,
        AVG(CASE WHEN ps.season_num <= 3 THEN ps.fg_pct END) AS early_fg_pct,
        AVG(CASE WHEN ps.season_num <= 3 THEN ps.avg_min END) AS early_min
    FROM player_seasons ps
    WHERE ps.season_num <= 3
    GROUP BY ps.player_id, ps.full_name, ps.position, ps.draft_round, ps.draft_number
)

SELECT
    ec.*,
    p.peak_ppg
FROM early_career ec
JOIN peak p ON ec.player_id = p.player_id
WHERE ec.early_pts IS NOT NULL
  AND p.peak_ppg IS NOT NULL
"""

peak_df = conn.sql(peak_sql).pl()
print(f"Peak prediction dataset: {peak_df.shape[0]:,} players")
print(f"Peak PPG range: {peak_df['peak_ppg'].min():.1f} - {peak_df['peak_ppg'].max():.1f}")
peak_df.head(5)

In [ ]:
# Encode position as numeric
pos_encode = {"G": 0, "PG": 0, "SG": 1, "G-F": 1, "F": 2, "SF": 2, "PF": 3, "F-G": 2, "F-C": 3, "C": 4, "C-F": 4}

model_peak_df = peak_df.with_columns(
    pl.col("position").replace(pos_encode, default=2).cast(pl.Float64).alias("pos_numeric"),
    pl.col("draft_round").fill_null(3).cast(pl.Float64),
    pl.col("draft_number").fill_null(60).cast(pl.Float64),
).drop_nulls(subset=["early_pts", "early_ast", "early_reb", "early_fg_pct", "early_min"])

feature_cols = ["early_pts", "early_ast", "early_reb", "early_fg_pct", "early_min",
                "draft_round", "draft_number", "pos_numeric"]

X = model_peak_df.select(feature_cols).to_pandas().values
y = model_peak_df["peak_ppg"].to_numpy()

# RandomForest with 5-fold CV
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)

cv_r2 = cross_val_score(rf, X, y, cv=5, scoring="r2")
cv_neg_mse = cross_val_score(rf, X, y, cv=5, scoring="neg_mean_squared_error")
cv_rmse = np.sqrt(-cv_neg_mse)

print(f"5-Fold CV Results:")
print(f"  R-squared: {cv_r2.mean():.3f} +/- {cv_r2.std():.3f}")
print(f"  RMSE:      {cv_rmse.mean():.2f} +/- {cv_rmse.std():.2f} PPG")

# Fit on full data for feature importance
rf.fit(X, y)
importances = rf.feature_importances_

# Feature importance bar chart
feat_labels = ["Early PPG", "Early APG", "Early RPG", "Early FG%", "Early MPG",
               "Draft Round", "Draft Number", "Position"]
sorted_idx = np.argsort(importances)[::-1]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[feat_labels[i] for i in sorted_idx],
    y=[importances[i] for i in sorted_idx],
    marker_color=[
        COLORS["gold"] if importances[i] == importances.max() else COLORS["purple"]
        for i in sorted_idx
    ],
    text=[f"{importances[i]:.3f}" for i in sorted_idx],
    textposition="outside",
))

fig.update_layout(
    template=TEMPLATE,
    title=f"Feature Importance for Peak PPG Prediction (R2={cv_r2.mean():.3f}, RMSE={cv_rmse.mean():.2f})",
    xaxis_title="Feature",
    yaxis_title="Importance",
    height=450,
    showlegend=False,
)
fig.show()

takeaway(
    f"The RandomForest achieves R-squared = {cv_r2.mean():.3f} and RMSE = {cv_rmse.mean():.2f} PPG. "
    "Early-career scoring (PPG in first 3 seasons) is by far the most important "
    "predictor of peak scoring -- unsurprisingly, players who score early tend to "
    "score at their peak. Minutes played and draft position also contribute, "
    "reflecting opportunity and pedigree."
)

<a id="7"></a>
## 7. Active Player Projections

We fit individual Gaussian aging curves to current stars using their career data
so far, then project the next 5 seasons. The uncertainty band widens as we
extrapolate further into the future.

In [ ]:
# Select 5 current stars with enough career data
projection_stars = ["LeBron James", "Stephen Curry", "Kevin Durant", "Giannis Antetokounmpo", "Jayson Tatum"]

# Verify they exist and have data
active_stars = []
for name in projection_stars:
    player_data = age_range.filter(pl.col("player_name") == name)
    if player_data.shape[0] >= 4:
        active_stars.append(name)
    else:
        print(f"Skipping {name}: only {player_data.shape[0]} qualifying seasons")

if not active_stars:
    # Fallback: pick the 5 active players with the most seasons
    active_counts = (
        age_range.filter(pl.col("is_active") == True)
        .group_by("player_name")
        .agg(pl.col("season_year").count().alias("n_seasons"))
        .sort("n_seasons", descending=True)
        .head(5)
    )
    active_stars = active_counts["player_name"].to_list()
    print(f"Using fallback stars: {active_stars}")

print(f"Projecting for: {active_stars}")

In [ ]:
proj_colors = [COLORS["purple"], COLORS["gold"], COLORS["blue"], COLORS["green"], COLORS["red"]]

fig = go.Figure()

for i, player_name in enumerate(active_stars):
    player_data = age_range.filter(
        pl.col("player_name") == player_name
    ).sort("player_age")

    x_player = player_data["player_age"].to_numpy().astype(float)
    y_player = player_data["avg_pts"].to_numpy().astype(float)
    color = proj_colors[i % len(proj_colors)]

    # Actual data (solid line)
    fig.add_trace(go.Scatter(
        x=x_player.tolist(), y=y_player.tolist(),
        mode="lines+markers",
        line=dict(color=color, width=3),
        marker=dict(size=7),
        name=f"{player_name} (actual)",
        legendgroup=player_name,
        hovertemplate=f"{player_name}<br>Age %{{x}}<br>PPG: %{{y:.1f}}<extra></extra>",
    ))

    # Fit individual Gaussian and project
    try:
        popt, pcov = curve_fit(
            gaussian_aging, x_player, y_player,
            p0=[y_player.max() - y_player.min(), 27, 5, y_player.min()],
            bounds=([0, 18, 1, 0], [50, 40, 20, 30]),
            maxfev=10000,
        )
        perr = np.sqrt(np.diag(pcov))

        # Project next 5 years
        max_age = int(x_player.max())
        future_ages = np.arange(max_age, max_age + 6, dtype=float)
        future_ppg = gaussian_aging(future_ages, *popt)

        # Uncertainty grows with distance from data
        distance_from_data = future_ages - max_age
        base_err = np.sqrt(mean_squared_error(y_player, gaussian_aging(x_player, *popt)))
        uncertainty = base_err * (1 + 0.3 * distance_from_data)

        upper = future_ppg + uncertainty
        lower = np.maximum(0, future_ppg - uncertainty)

        # Projected line (dashed)
        fig.add_trace(go.Scatter(
            x=future_ages.tolist(), y=future_ppg.tolist(),
            mode="lines",
            line=dict(color=color, width=2, dash="dash"),
            name=f"{player_name} (projected)",
            legendgroup=player_name,
            hovertemplate=f"{player_name} proj<br>Age %{{x}}<br>PPG: %{{y:.1f}}<extra></extra>",
        ))

        # Confidence interval
        fig.add_trace(go.Scatter(
            x=future_ages.tolist(), y=upper.tolist(),
            mode="lines", line=dict(width=0),
            showlegend=False, hoverinfo="skip",
            legendgroup=player_name,
        ))
        fig.add_trace(go.Scatter(
            x=future_ages.tolist(), y=lower.tolist(),
            mode="lines", line=dict(width=0),
            fill="tonexty",
            fillcolor=color.replace("#", "").join(["rgba(", ",0.15)"]) if False else f"rgba(100,100,100,0.1)",
            showlegend=False, hoverinfo="skip",
            legendgroup=player_name,
        ))

        print(f"{player_name}: peak_age={popt[1]:.1f}, current_age={max_age}, "
              f"projected next year: {future_ppg[1]:.1f} PPG")

    except RuntimeError:
        print(f"{player_name}: curve_fit failed, skipping projection")

fig.update_layout(
    template=TEMPLATE,
    title="Active Player Scoring Projections: Actual (Solid) + Projected (Dashed)",
    xaxis_title="Player Age",
    yaxis_title="Points Per Game",
    height=600,
    legend=dict(font=dict(size=10)),
)
fig.show()

takeaway(
    "Individual aging curves reveal different trajectories. LeBron's remarkably "
    "flat curve projects continued high-level production, while younger stars like "
    "Tatum and Giannis may still be approaching their peak. The widening uncertainty "
    "bands reflect the inherent unpredictability of long-range projections -- injuries, "
    "role changes, and team context all create variance that a purely age-based model "
    "cannot capture."
)

<a id="8"></a>
## 8. Aging Gracefully Index

Which players most defied their expected decline? For every player-season at age
30 or older, we compare their actual PPG against the population-expected PPG at
that age (from our Gaussian model), then rank who most exceeded expectations.

In [ ]:
# Compute expected PPG at each age from the fitted model
if "Points" in fit_results:
    pts_popt = fit_results["Points"]["popt"]

    # For players age 30+, compute aging_grace_index = actual / expected
    veterans = age_range.filter(
        pl.col("player_age") >= 30
    ).with_columns(
        pl.col("player_age").cast(pl.Float64).alias("age_float")
    )

    # Compute expected PPG from the Gaussian model
    expected_ppg = gaussian_aging(
        veterans["age_float"].to_numpy(),
        *pts_popt
    )

    veterans = veterans.with_columns(
        pl.Series("expected_ppg", expected_ppg),
    ).with_columns(
        (pl.col("avg_pts") / pl.col("expected_ppg").clip(lower_bound=1.0)).alias("aging_grace_index")
    )

    # Average the index across all age-30+ seasons per player
    grace_ranking = (
        veterans
        .group_by(["player_id", "player_name"])
        .agg([
            pl.col("aging_grace_index").mean().alias("avg_grace_index"),
            pl.col("avg_pts").mean().alias("avg_ppg_after_30"),
            pl.col("expected_ppg").mean().alias("avg_expected_ppg"),
            pl.col("player_age").max().alias("max_age"),
            pl.col("player_age").count().alias("seasons_30_plus"),
        ])
        .filter(pl.col("seasons_30_plus") >= 2)  # at least 2 seasons 30+
        .sort("avg_grace_index", descending=True)
    )

    top20 = grace_ranking.head(20)
    print("Top 20 'Aging Gracefully' Players (highest actual/expected PPG ratio after 30):")
    for row in top20.iter_rows(named=True):
        print(f"  {row['player_name']:25s} grace={row['avg_grace_index']:.2f}  "
              f"actual={row['avg_ppg_after_30']:.1f}  expected={row['avg_expected_ppg']:.1f}  "
              f"max_age={row['max_age']}  seasons_30+={row['seasons_30_plus']}")
else:
    print("Points model not fitted. Cannot compute aging grace index.")
    top20 = None

In [ ]:
if top20 is not None:
    # Horizontal bar chart
    fig = go.Figure()

    names = top20["player_name"].to_list()[::-1]
    values = top20["avg_grace_index"].to_list()[::-1]
    actual_ppg = top20["avg_ppg_after_30"].to_list()[::-1]

    fig.add_trace(go.Bar(
        y=names,
        x=values,
        orientation="h",
        marker=dict(
            color=values,
            colorscale=[[0, COLORS["gray"]], [0.5, COLORS["gold"]], [1, COLORS["green"]]],
            line=dict(color=COLORS["purple"], width=1),
        ),
        text=[f"{v:.2f} ({ppg:.1f} PPG)" for v, ppg in zip(values, actual_ppg)],
        textposition="outside",
        hovertemplate="%{y}<br>Grace Index: %{x:.2f}<extra></extra>",
    ))

    # Reference line at 1.0 (exactly meeting expectations)
    fig.add_vline(x=1.0, line_dash="dash", line_color=COLORS["red"],
                  annotation_text="Expected", annotation_position="top right")

    fig.update_layout(
        template=TEMPLATE,
        title="Aging Gracefully Index: Who Most Defied Expected Decline After 30?",
        xaxis_title="Aging Grace Index (Actual PPG / Expected PPG)",
        height=650,
        margin=dict(l=180),
    )
    fig.show()

    takeaway(
        "Players who age most gracefully tend to share common traits: elite "
        "shooting ability, high basketball IQ, and skill-based games that do not "
        "rely on raw athleticism. LeBron, Curry, and Durant exemplify how "
        "perimeter shooting and playmaking translate to longevity, while players "
        "dependent on explosiveness (rim-running centers, slashing guards) tend "
        "to decline closer to the population average. The lesson: invest in "
        "skills that do not age."
    )

<a id="9"></a>
## 9. Conclusion & Cross-Links

In [ ]:
# Cleanup handled by nbadb_utils.get_connection() via atexit
print("Analysis complete.")

### Summary

This notebook modeled NBA career trajectories through multiple lenses:

1. **Raw aging curves** show scoring peaks ~27, playmaking ~28-29, and athleticism-based stats decline earlier
2. **Position-specific analysis** reveals guards maintain playmaking longest, while centers decline in scoring earliest
3. **Parametric Gaussian model** quantifies peak age and decline rate for each stat category
4. **Era comparison** suggests modern players score more at every age, though peak timing is similar
5. **Peak prediction** from early-career stats achieves meaningful R-squared via RandomForest
6. **Active projections** fit individual curves to forecast near-term production
7. **Aging gracefully index** identifies who most defies expected decline -- skill-based players dominate

---

### nbadb Notebook Suite

| Part | Notebook | Description |
|---|---|---|
| Part 1 | [MVP Prediction](./nba_mvp_predictor.ipynb) | MVP Prediction with Tracking & Synergy Data |
| Part 2 | [Data-Driven Player Archetypes](./nba_player_archetypes.ipynb) | Data-Driven Player Archetypes (UMAP + GMM) |
| Part 3 | [Game Outcome Prediction](./nba_game_prediction.ipynb) | Game Outcome Prediction (Stacking Ensemble) |
| Part 4 | [Draft Combine to Career](./nba_draft_combine_analysis.ipynb) | Draft Combine to Career Prediction |
| Part 5 | [Defense Decoded](./nba_defense_decoded.ipynb) | Defense Decoded (Tracking + Hustle + Synergy PCA) |
| Part 6 | [Interactive Player Explorer](./nba_player_dashboard.ipynb) | Interactive Player Explorer |
| Part 7 | [Spatial Shot Analysis](./nba_shot_chart_analysis.ipynb) | Spatial Shot Analysis & 3-Point Revolution |
| Part 8 | [Player Similarity Engine](./nba_player_similarity.ipynb) | Player Similarity Engine (Cosine + Manhattan) |
| **Part 9** | **Aging Curves** (this notebook) | **Career Trajectory & Aging Curve Modeling** |
| Part 10 | [Play-by-Play Insights](./nba_play_by_play_insights.ipynb) | Play-by-Play: Win Probability, Runs & Clutch |

---

**Dataset**: [wyattowalsh/basketball on Kaggle](https://www.kaggle.com/datasets/wyattowalsh/basketball)
**Documentation**: [nbadb.dev](https://nbadb.dev)
**Source**: [github.com/wyattowalsh/nbadb](https://github.com/wyattowalsh/nbadb)